In [25]:
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage.feature import graycomatrix, graycoprops
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
import os
import random


In [26]:
def set_seed(seed=999):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [27]:
'''
def load_frozen_dataset(
    base_path='/kaggle/input/datasets/aaryaupi/ct3-dataset',
    data_root='/kaggle/input/datasets/aaryaupi/ct3-dataset'
):
    train_df = pd.read_csv(os.path.join(base_path, 'train.csv'))
    val_df   = pd.read_csv(os.path.join(base_path, 'val.csv'))
    test_df  = pd.read_csv(os.path.join(base_path, 'test.csv'))

    for df in [train_df, val_df, test_df]:
        for col in ['ct_path', 'mu_path', 'mask_path']:
            if col not in df.columns:
                continue
            # Fix duplicated subdirectory names
            df[col] = df[col].str.replace('/ct_processed/ct_processed/',     '/ct_processed/',     regex=False)
            df[col] = df[col].str.replace('/mu_processed/mu_processed/',     '/mu_processed/',     regex=False)
            df[col] = df[col].str.replace('/mask_processed/mask_processed/', '/mask_processed/',   regex=False)
            # Remap any old /kaggle/working/ or /kaggle/input/output-data/ prefixes
            df[col] = df[col].str.replace('/kaggle/working/',             data_root + '/', regex=False)
            df[col] = df[col].str.replace('/kaggle/input/output-data/',   data_root + '/', regex=False)

    return train_df, val_df, test_df


# ─────────────────────────────────────────────
# 3. CT LOADING — matches ARSIVAE: no resize,
#    raw .npy, assumed (512, 512)
# ─────────────────────────────────────────────

def load_ct(path):
    ct = np.load(path).astype(np.float32)

    # Volume (D, H, W) → middle slice
    if ct.ndim == 3:
        ct = ct[ct.shape[0] // 2]
    elif ct.ndim != 2:
        raise ValueError(f"Unexpected ndim={ct.ndim} at {path}")

    # Normalise to [0, 1] — same implicit range ARSIVAE uses
    lo, hi = ct.min(), ct.max()
    ct = (ct - lo) / (hi - lo + 1e-8)

    return ct[np.newaxis]   # (1, H, W)


# ─────────────────────────────────────────────
# 4. DATASET — CNN only needs ct + label
# ─────────────────────────────────────────────

class CTDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df      = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        ct    = load_ct(row['ct_path'])          # (1, H, W)
        label = int(row['label'])
        ct    = torch.tensor(ct, dtype=torch.float32)

        if self.augment:
            if random.random() > 0.5:
                ct = torch.flip(ct, dims=[2])    # horizontal flip
            if random.random() > 0.5:
                ct = torch.flip(ct, dims=[1])    # vertical flip

        return {
            'ct':    ct,
            'label': torch.tensor(label, dtype=torch.long)
        }

'''


'\ndef load_frozen_dataset(\n    base_path=\'/kaggle/input/datasets/aaryaupi/ct3-dataset\',\n    data_root=\'/kaggle/input/datasets/aaryaupi/ct3-dataset\'\n):\n    train_df = pd.read_csv(os.path.join(base_path, \'train.csv\'))\n    val_df   = pd.read_csv(os.path.join(base_path, \'val.csv\'))\n    test_df  = pd.read_csv(os.path.join(base_path, \'test.csv\'))\n\n    for df in [train_df, val_df, test_df]:\n        for col in [\'ct_path\', \'mu_path\', \'mask_path\']:\n            if col not in df.columns:\n                continue\n            # Fix duplicated subdirectory names\n            df[col] = df[col].str.replace(\'/ct_processed/ct_processed/\',     \'/ct_processed/\',     regex=False)\n            df[col] = df[col].str.replace(\'/mu_processed/mu_processed/\',     \'/mu_processed/\',     regex=False)\n            df[col] = df[col].str.replace(\'/mask_processed/mask_processed/\', \'/mask_processed/\',   regex=False)\n            # Remap any old /kaggle/working/ or /kaggle/input/out

In [28]:
def load_frozen_dataset(
    base_path='/kaggle/input/datasets/aaryaupi/ct3-dataset',
    data_root='/kaggle/input/datasets/aaryaupi/ct3-dataset'
):
    train_df = pd.read_csv(os.path.join(base_path, 'train.csv'))
    val_df   = pd.read_csv(os.path.join(base_path, 'val.csv'))
    test_df  = pd.read_csv(os.path.join(base_path, 'test.csv'))

    for df in [train_df, val_df, test_df]:
        for col in ['ct_path', 'mu_path', 'mask_path']:
            if col not in df.columns:
                continue
            # Fix duplicated subdirectory names (all variants)
            df[col] = df[col].str.replace('/ct_processed/ct_processed/',     '/ct_processed/',   regex=False)
            df[col] = df[col].str.replace('/mu_processed/mu_processed/',     '/mu_processed/',   regex=False)
            df[col] = df[col].str.replace('/mask_processed/mask_processed/', '/mask_processed/', regex=False)
            df[col] = df[col].str.replace('/output_data/output_data/',       '/output_data/',    regex=False)

            # Remap old prefixes to data_root
            df[col] = df[col].str.replace('/kaggle/working/',           data_root + '/', regex=False)
            df[col] = df[col].str.replace('/kaggle/input/output-data/', data_root + '/', regex=False)

            # Remap ct_processed → output_data since that's the actual folder name
            df[col] = df[col].str.replace('/ct_processed/',   '/output_data/', regex=False)
            df[col] = df[col].str.replace('/mask_processed/', '/output_data/', regex=False)
            df[col] = df[col].str.replace('/mu_processed/',   '/output_data/', regex=False)

    return train_df, val_df, test_df

In [29]:

# ─────────────────────────────────────────────
# 3. CT LOADING — matches ARSIVAE: no resize,
#    raw .npy, assumed (512, 512)
# ─────────────────────────────────────────────

def load_ct(path):
    ct = np.load(path).astype(np.float32)

    # Volume (D, H, W) → middle slice
    if ct.ndim == 3:
        ct = ct[ct.shape[0] // 2]
    elif ct.ndim != 2:
        raise ValueError(f"Unexpected ndim={ct.ndim} at {path}")

    # Normalise to [0, 1] — same implicit range ARSIVAE uses
    lo, hi = ct.min(), ct.max()
    ct = (ct - lo) / (hi - lo + 1e-8)

    return ct[np.newaxis]   # (1, H, W)


# ─────────────────────────────────────────────
# 4. DATASET — CNN only needs ct + label
# ─────────────────────────────────────────────

class CTDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df      = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        ct    = load_ct(row['ct_path'])          # (1, H, W)
        label = int(row['label'])
        ct    = torch.tensor(ct, dtype=torch.float32)

        if self.augment:
            if random.random() > 0.5:
                ct = torch.flip(ct, dims=[2])    # horizontal flip
            if random.random() > 0.5:
                ct = torch.flip(ct, dims=[1])    # vertical flip

        return {
            'ct':    ct,
            'label': torch.tensor(label, dtype=torch.long)
        }



In [30]:

class CNNClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Identical conv stack to ARSIVAE Encoder
        self.features = nn.Sequential(
            nn.Conv2d(1,   32,  4, 2, 1), nn.BatchNorm2d(32),  nn.LeakyReLU(0.2),
            nn.Conv2d(32,  64,  4, 2, 1), nn.BatchNorm2d(64),  nn.LeakyReLU(0.2),
            nn.Conv2d(64,  128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d((4, 4))  # → (512, 4, 4)
        )
        self.classifier = nn.Sequential(
            nn.Linear(512 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out',
                                        nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)



def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(loader, desc='Train')
    for batch in pbar:
        x = batch['ct'].to(device)
        y = batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss   = F.cross_entropy(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        preds      = logits.argmax(dim=1)
        correct    += (preds == y).sum().item()
        total      += y.size(0)
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.3f}',
                          'acc':  f'{correct/total:.3f}'})
    return total_loss / len(loader), correct / total


def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            x = batch['ct'].to(device)
            y = batch['label'].to(device)
            logits     = model(x)
            loss       = F.cross_entropy(logits, y)
            total_loss += loss.item()
            probs      = F.softmax(logits, dim=1)[:, 1]
            preds      = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='binary')
    auc = roc_auc_score(all_labels, all_probs)
    cm  = confusion_matrix(all_labels, all_preds)
    return total_loss / len(loader), acc, f1, auc, cm



def plot_training_curves(history, save_path='cnn_training_curves.png'):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train')
    axes[0].plot(epochs, history['val_loss'],   'r-', label='Val')
    axes[0].set_title('Loss');     axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(epochs, history['train_acc'],  'b-', label='Train')
    axes[1].plot(epochs, history['val_acc'],    'r-', label='Val')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[2].plot(epochs, history['val_auc'],    'g-', label='Val AUC')
    axes[2].set_title('AUC');      axes[2].legend(); axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()



def train(model, train_loader, val_loader, device, epochs=50):
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    history   = {'train_loss': [], 'val_loss': [], 'train_acc': [],
                 'val_acc': [], 'val_f1': [], 'val_auc': []}
    best_val_auc = 0
    best_epoch   = 0

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, device)
        val_loss, val_acc, val_f1, val_auc, _ = evaluate(model, val_loader, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['val_auc'].append(val_auc)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} | "
              f"ValLoss={val_loss:.4f} ValAcc={val_acc:.4f} "
              f"ValF1={val_f1:.4f} ValAUC={val_auc:.4f}")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch   = epoch + 1
            torch.save(model.state_dict(), 'best_cnn.pth')

    print(f"Best epoch: {best_epoch} | Best Val AUC: {best_val_auc:.4f}")
    return model, history

In [31]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import json
import glob

def main():
    SEED = 16          # matches ARSIVAE
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"CNN BASELINE - SEED {SEED} | Device: {device}")

    train_df, val_df, test_df = load_frozen_dataset(
        base_path='/kaggle/input/datasets/aaryaupi/ct3-dataset',
        data_root='/kaggle/input/datasets/aaryaupi/ct3-dataset'
    )

    print(f"Sizes — Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

    # Sanity check first file
    first = train_df['ct_path'].iloc[0]
    print(f"First CT path : {first}")
    print(f"File exists   : {os.path.exists(first)}")

    # Confirm what's actually on disk
    samples = glob.glob('/kaggle/input/datasets/aaryaupi/ct3-dataset/output_data/*.npy')[:3]
    print(f"Sample files in output_data: {samples}")

    # Hard stop if paths are broken — don't waste a training run
    if not os.path.exists(first):
        raise FileNotFoundError(
            f"CT file not found: {first}\n"
            f"Check the glob output above and fix load_frozen_dataset() paths."
        )

    train_dataset = CTDataset(train_df, augment=True)
    val_dataset   = CTDataset(val_df,   augment=False)
    test_dataset  = CTDataset(test_df,  augment=False)

    train_loader = DataLoader(
        train_dataset, batch_size=32, shuffle=True,
        num_workers=4, pin_memory=True,
        drop_last = True,
        worker_init_fn=lambda wid: np.random.seed(SEED + wid)
    )
    val_loader  = DataLoader(val_dataset,  batch_size=32, shuffle=False,
                             num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False,
                             num_workers=4, pin_memory=True)

    model = CNNClassifier(num_classes=2).to(device)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {total_params:,}")

    model, history = train(model, train_loader, val_loader, device, epochs=50)

    model.load_state_dict(torch.load('best_cnn.pth'))
    _, val_acc,  val_f1,  val_auc,  val_cm  = evaluate(model, val_loader,  device)
    _, test_acc, test_f1, test_auc, test_cm = evaluate(model, test_loader, device)

    print(f"\nVal  | Acc={val_acc:.4f}  F1={val_f1:.4f}  AUC={val_auc:.4f}")
    print(f"Test | Acc={test_acc:.4f} F1={test_f1:.4f} AUC={test_auc:.4f}")
    print(f"Val  Confusion Matrix:\n{val_cm}")
    print(f"Test Confusion Matrix:\n{test_cm}")

    plot_training_curves(history, 'cnn_training_curves.png')

    results = {
        'seed': SEED,
        'val_acc':  float(val_acc),  'val_f1':  float(val_f1),  'val_auc':  float(val_auc),
        'test_acc': float(test_acc), 'test_f1': float(test_f1), 'test_auc': float(test_auc),
        'val_confusion_matrix':  val_cm.tolist(),
        'test_confusion_matrix': test_cm.tolist(),
        'best_val_auc': float(max(history['val_auc'])),
        'history': {k: [float(v) for v in vals] for k, vals in history.items()}
    }

    with open(f'cnn_results_seed_{SEED}.json', 'w') as f:
        json.dump(results, f, indent=2)

    print(f"\nDone. Results saved to cnn_results_seed_{SEED}.json")
    return model, history


if __name__ == '__main__':
    model, history = main()

CNN BASELINE - SEED 16 | Device: cuda
Sizes — Train: 1281  Val: 261  Test: 278
First CT path : /kaggle/input/datasets/aaryaupi/ct3-dataset/output_data/mosmed_normal_0000_slice_010_ct.npy
File exists   : True
Sample files in output_data: ['/kaggle/input/datasets/aaryaupi/ct3-dataset/output_data/mosmed_normal_0030_slice_018_ct.npy', '/kaggle/input/datasets/aaryaupi/ct3-dataset/output_data/mosmed_normal_0020_slice_021_mu.npy', '/kaggle/input/datasets/aaryaupi/ct3-dataset/output_data/mosmed_covid_0006_slice_018_mu.npy']
Trainable parameters: 7,050,786


Train: 100%|██████████| 40/40 [00:04<00:00,  8.52it/s, loss=2.466, acc=0.658]


Epoch 1/50 | TrainLoss=3.4354 TrainAcc=0.6578 | ValLoss=4.6406 ValAcc=0.5479 ValF1=0.6609 ValAUC=0.5485


Train: 100%|██████████| 40/40 [00:04<00:00,  8.52it/s, loss=2.484, acc=0.738]


Epoch 2/50 | TrainLoss=2.3933 TrainAcc=0.7375 | ValLoss=4.6017 ValAcc=0.5709 ValF1=0.5372 ValAUC=0.6021


...

Epoch 48/50 | TrainLoss=0.0250 TrainAcc=0.9922 | ValLoss=6.6252 ValAcc=0.5709 ValF1=0.5410 ValAUC=0.6151


Train: 100%|██████████| 40/40 [00:04<00:00,  8.44it/s, loss=0.000, acc=0.991]


Epoch 49/50 | TrainLoss=0.0321 TrainAcc=0.9906 | ValLoss=6.2653 ValAcc=0.5785 ValF1=0.5528 ValAUC=0.6216


Train: 100%|██████████| 40/40 [00:04<00:00,  8.65it/s, loss=0.000, acc=0.993]


Epoch 50/50 | TrainLoss=0.0386 TrainAcc=0.9930 | ValLoss=6.6223 ValAcc=0.5670 ValF1=0.5311 ValAUC=0.6150
Best epoch: 7 | Best Val AUC: 0.7175

Val  | Acc=0.5670  F1=0.4593  AUC=0.7175
Test | Acc=0.6763 F1=0.6341 AUC=0.7455
Val  Confusion Matrix:
[[100  32]
 [ 81  48]]
Test Confusion Matrix:
[[110  23]
 [ 67  78]]

Done. Results saved to cnn_results_seed_16.json
